In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# ──── Paths ──────────────────────────────────────────────────────
TEST_TARGET    = '../data/processed/y_test.csv'
STACKED_PROBS  = '../notebooks/stacked_probabilities.csv'

# ──── Load data ──────────────────────────────────────────────────
y_test = pd.read_csv(TEST_TARGET)['TARGET']
probs  = pd.read_csv(STACKED_PROBS)['stacked_prob']

In [ ]:
# ──── Cost matrix (adjust values based on business context) ───────
# FP = approve bad loan → high cost (loss of principal + interest)
# FN = reject good loan → opportunity cost (lost interest)
COST_FP = 12000     # cost of false positive (approve defaulter)
COST_FN =  2500     # cost of false negative (reject good applicant)

def expected_cost(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return fp * COST_FP + fn * COST_FN

In [ ]:
# ──── Evaluate many thresholds ───────────────────────────────────
thresholds = np.arange(0.02, 0.50, 0.01)   # finer grid near low values (imbalanced data)
costs = []

for thresh in thresholds:
    y_pred = (probs >= thresh).astype(int)
    cost = expected_cost(y_test, y_pred)
    costs.append(cost)

costs = np.array(costs)

In [ ]:
# ──── Find optimal threshold ─────────────────────────────────────
best_idx = np.argmin(costs)
optimal_threshold = thresholds[best_idx]
min_cost = costs[best_idx]

print(f"Optimal threshold : {optimal_threshold:.3f}")
print(f"Minimum expected cost: {min_cost:,.0f}")

# Compare with default 0.5
default_pred = (probs >= 0.5).astype(int)
default_cost = expected_cost(y_test, default_pred)
print(f"Cost @ 0.5 threshold : {default_cost:,.0f}")
print(f"Cost savings         : {default_cost - min_cost:,.0f}")

In [ ]:
# ──── Plot cost vs threshold ─────────────────────────────────────
plt.figure(figsize=(10,6))
plt.plot(thresholds, costs, marker='o', markersize=4)
plt.axvline(optimal_threshold, color='red', linestyle='--', alpha=0.7,
            label=f'Optimal = {optimal_threshold:.3f}')
plt.xlabel('Decision Threshold (probability of default)')
plt.ylabel('Total Expected Cost')
plt.title('Cost-Optimized Decision Threshold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('../reports/figures/05_cost_vs_threshold.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save optimal value
with open('../notebooks/optimal_threshold.txt', 'w') as f:
    f.write(str(optimal_threshold))

print("Optimal threshold saved to: ../notebooks/optimal_threshold.txt")